# Parameter Search and Density-Aware Evaluation

This notebook documents the current fundus-only stage of the retinal vasculature generative modeling project. It should be read after the earlier baseline and image-constrained notebooks. OCTA data are not yet available, so this notebook focuses on parameter search, fair density evaluation, and terminal-over-density visualization for fundus images.

## Current Modeling Question

The previous density-aware model reduced over-pruning but still generated trees that were smaller than the baseline. The current goal is to recover spatial reach while keeping density-aware terminal placement active. In practice, the target is a density-aware model with roughly 20-25 terminal nodes and improved density responsiveness.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import Image, display, Markdown

results_dir = project_root / "results"
figures_dir = project_root / "figures"

## Latest Summary Files

The latest generated outputs are stored in `results/`. The most useful files are `latest_run_summary.md`, `best_parameter_summary.md`, `evaluation_summary.csv`, and `parameter_search.csv`.

In [ ]:
for path in [
    results_dir / "latest_run_summary.md",
    results_dir / "best_parameter_summary.md",
]:
    if path.exists():
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print(f"Missing: {path}")

## Evaluation Summary

The table below compares baseline, image-constrained, density ablations, and the full density-aware model over 15 fundus images.

In [ ]:
summary_path = results_dir / "evaluation_summary.csv"
summary = pd.read_csv(summary_path)
summary

## Parameter Search

The focused parameter search tested 96 combinations. The selected setting prioritizes recovering terminal count and occupied grid coverage while keeping density direction guidance active.

In [ ]:
search_path = results_dir / "parameter_search.csv"
search = pd.read_csv(search_path)
search.sort_values("search_score", ascending=False).head(10)

## Selected Full Density-Aware Parameters

The selected parameters are:

- `alpha = 0.76`
- `max_depth = 7`
- `initial_length = 0.23`
- `density_weight = 0.60`
- `density_depth_weight = 0.75`
- `density_direction_weight = 0.70`
- `density_survival_weight = 0.00`

This setting increases the full density-aware model from the previous 14.867 average terminals to 20.733 average terminals.

## Fair Density Metrics

Two metrics were added to avoid misleading comparisons caused by different terminal counts:

- `matched_terminal_density_score`: density score after sampling models down to the same terminal count.
- `density_lift_over_random`: matched density score compared with uniformly random retinal points.

The current density lift remains negative, so density matching is still a weakness even after spatial reach improves.

In [ ]:
cols = [
    "model", "terminals", "occupied_grid_coverage",
    "terminal_density_score", "matched_terminal_density_score",
    "density_lift_over_random"
]
summary[cols]

## Model Comparison Figure

In [ ]:
display(Image(filename=str(figures_dir / "fig2_model_comparison.png")))

## Quantitative Evaluation Figure

In [ ]:
display(Image(filename=str(figures_dir / "fig4_evaluation_summary.png")))

## Terminal Nodes Over Density Map

This figure overlays generated terminal nodes directly on the fundus-derived density map. It makes the image-specific behavior easier to inspect qualitatively.

In [ ]:
display(Image(filename=str(figures_dir / "fig5_terminal_density_overlay.png")))

## Interpretation

The latest parameter search improves spatial reach. The full density-aware model now averages 20.733 terminals and occupied grid coverage of 0.037, compared with 14.867 terminals and 0.027 occupied grid coverage in the previous calibrated run.

The main tradeoff is that terminal density score decreases from 0.217 to 0.187. This means the model reaches more of the retinal field, but terminals are less concentrated in the highest-density regions. The next stage should improve density matching while preserving the recovered terminal count.

## Next Technical Step

The next modeling step should refine density-guided direction selection. Instead of selecting only among nearby candidate angles, branch candidates should be evaluated by whether they move toward increasing density over a short spatial horizon. This would target density matching without returning to aggressive survival pruning.

## Reproducing Results

Run these scripts from the repository root after placing the 15 fundus images in `data/raw/healthy/`:

```bash
python run_parameter_search.py
python run_results.py
```